# Parsing code

Inspect Onnx

In [3]:
import onnx

# Load model
onnx_model_name = "yolov11s"
model_path = f"../models/original/{onnx_model_name}.onnx"
model = onnx.load(model_path)

# Inspect inputs
input_tensor = model.graph.input[0]
input_name = input_tensor.name

# Check  dimensions
shape = [d.dim_value for d in input_tensor.type.tensor_type.shape.dim]

print(f"Model loaded")
print(f"Input name: '{input_name}'")
print(f"Dimensions: {shape}")

chosen_hw_arch = "hailo8"
onnx_path = f"../models/original/{onnx_model_name}.onnx"

Model loaded
Input name: 'images'
Dimensions: [1, 3, 640, 640]


Create .har file using hailo SDK

### YOLOV11

In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from hailo_sdk_client import ClientRunner

runner = ClientRunner(hw_arch=chosen_hw_arch)

end_node_names = [
    "/model.23/cv2.0/cv2.0.2/Conv",
    "/model.23/cv3.0/cv3.0.2/Conv",
    "/model.23/cv2.1/cv2.1.2/Conv",
    "/model.23/cv3.1/cv3.1.2/Conv",
    "/model.23/cv2.2/cv2.2.2/Conv",
    "/model.23/cv3.2/cv3.2.2/Conv"
]


try:
    hn, npz = runner.translate_onnx_model(
        onnx_path,
        onnx_model_name,
        start_node_names=["images"],
        end_node_names=end_node_names,
        net_input_shapes={"images": [1, 3, 640, 640]},
    )
    print("Model translation succesful")
except Exception as  e:
    print(f"Error during model  transñation: {e}")
    raise

hailo_model_har_name = f"../models/intermediate/{onnx_model_name}_hailo_model.har"
try:
    runner.save_har(hailo_model_har_name)
    print(f"HAR file saved as: {hailo_model_har_name}")
except Exception as e:
    print(f"Error saving HAR file: {e}")

[info] Translation started on ONNX model yolov11s
[info] Restored ONNX model yolov11s (completion time: 00:00:00.26)
[info] Extracted ONNXRuntime meta-data for Hailo model (completion time: 00:00:01.21)
[info] Start nodes mapped from original model: 'images': 'yolov11s/input_layer1'.
[info] End nodes mapped from original model: '/model.23/cv2.0/cv2.0.2/Conv', '/model.23/cv3.0/cv3.0.2/Conv', '/model.23/cv2.1/cv2.1.2/Conv', '/model.23/cv3.1/cv3.1.2/Conv', '/model.23/cv2.2/cv2.2.2/Conv', '/model.23/cv3.2/cv3.2.2/Conv'.
[info] Translation completed on ONNX model yolov11s (completion time: 00:00:03.95)
Model translation succesful
[info] Saved HAR to: /home/jainogue/hailo/models/intermediate/yolov11s_hailo_model.har
HAR file saved as: ../models/intermediate/yolov11s_hailo_model.har


Inspect .har file 

In [ ]:
from hailo_sdk_client import ClientRunner
onnx_model_name = "yolov11n"
# Load the HAR file
har_path = f"../models/intermediate/{onnx_model_name}_hailo_model.har"

runner = ClientRunner(har=har_path)
from pprint import pprint

try:
    # Access the HailoNet as an OrderedDict
    hn_dict = runner.get_hn()
    print("Inspecting layers from HailoNet (OrderedDict):")

    # Pretty-print each layer
    for key, value in hn_dict.items():
        print(f"Key: {key}")
        pprint(value)
        print("\n" + "="*80 + "\n")  # Add a separator between layers for clarity

except Exception as e:
    print(f"Error while inspecting hn_dict: {e}")

Create NMS pos-processing config

In [ ]:
import json
import os

# Updated NMS layer configuration dictionary
nms_layer_config_yolov11 = {
    "nms_scores_th": 0.2,
    "nms_iou_th": 0.7,
    "image_dims": [
        640,
        640
    ],
    "max_proposals_per_class": 100,
    "classes": 2,
    "regression_length": 16,
    "background_removal": False,
    "background_removal_index": 0,
    "bbox_decoders": [
        {
            "name": "bbox_decoder51",
            "stride": 8,
            "reg_layer": "conv51",
            "cls_layer": "conv54"
        },
        {
            "name": "bbox_decoder62",
            "stride": 16,
            "reg_layer": "conv62",
            "cls_layer": "conv65"
        },
        {
            "name": "bbox_decoder77",
            "stride": 32,
            "reg_layer": "conv77",
            "cls_layer": "conv80"
        }
    ]
}



# Path to save the updated JSON configuration
output_dir = "../tools/NMS"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "nms_layer_config_yolov8n.json")

# Save the updated configuration as a JSON file
with open(output_path, "w") as json_file:
    json.dump(nms_layer_config_yolov11, json_file, indent=4)

print(f"NMS layer configuration saved to {output_path}")